# 09. 제출 파일 비교 분석

## 비교 대상
| | 파일 | 형태 |
|---|---|---|
| **우리 모델** | `submission_order_products.csv` | order_id, product_id 행 단위 |
| **참조 (Faron)** | `Faron-opt_bagging-v3.csv` | order_id, products(공백 구분) |

## 주의
> `Faron-opt_bagging-v3`는 **다른 모델의 예측값**이지 정답(ground truth)이 아닙니다.  
> 따라서 아래 지표는 **"우리 예측이 Faron 예측과 얼마나 일치하는가"** 를 나타냅니다.  
> Faron 모델이 높은 public LB 점수를 가진 참조 모델이라고 가정하고 비교합니다.

## 평가 지표
- **Mean F1-Score** — Instacart 대회 공식 지표  
  `F1 = 2 × Precision × Recall / (Precision + Recall)` (주문별 계산 후 평균)
- **Precision** — 우리가 예측한 상품 중 Faron도 예측한 비율
- **Recall** — Faron이 예측한 상품 중 우리도 예측한 비율
- **Jaccard** — 합집합 대비 교집합 비율

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

_candidates = [
    Path(r'c:\Users\gksal\capstone-kpick\K-Pick'),
    Path.cwd(),
    Path.cwd().parent,
]
BASE = next((p for p in _candidates if (p / 'data' / 'prep').exists()), None)
if BASE is None:
    raise FileNotFoundError(f'data/prep 폴더를 찾을 수 없습니다. 현재 경로: {Path.cwd()}')

PREP = BASE / 'data' / 'prep'
print(f'PREP : {PREP}')
print('라이브러리 로드 완료')

---
## 1. 두 파일 로드 및 동일 형태로 변환
비교를 위해 둘 다 `{order_id → set(product_ids)}` 딕셔너리로 변환

In [ ]:
# ── 우리 모델 파일 로드 ──────────────────────────────────────────────
_our = PREP / 'submission_order_products.csv'
if not _our.exists():
    raise FileNotFoundError('submission_order_products.csv 없음 — 08_submission.ipynb를 먼저 실행하세요.')

our_df = pd.read_csv(_our)
print(f'로드 후 행 수: {len(our_df):,}')

# product_id를 숫자로 변환 (None/NaN → 결측값)
our_df['product_id'] = pd.to_numeric(our_df['product_id'], errors='coerce')

# astype(int) 대신 set comprehension 안에서 개별 변환
# → NaN 값은 dropna로 제거된 후 처리되므로 오류 없음
our_pred = (
    our_df.dropna(subset=['product_id'])
    .groupby('order_id')['product_id']
    .apply(lambda s: {int(v) for v in s})
    .to_dict()
)
print(f'우리 예측 주문 수: {len(our_pred):,}')

sample_oid = next(iter(our_pred))
print(f'샘플 (order {sample_oid}): {sorted(our_pred[sample_oid])[:5]}')


In [ ]:
# ── Faron 참조 파일 로드 ─────────────────────────────────────────────
_ref = PREP / 'Faron-opt_bagging-v3.csv'
ref_df = pd.read_csv(_ref)
print(f'Faron 파일 컬럼: {ref_df.columns.tolist()}')
print(f'Faron 파일 행 수: {len(ref_df):,}')

# "products" 컬럼: 공백 구분 문자열 → set 변환
# 'None' 문자열은 빈 set으로 처리
def parse_products(s):
    if pd.isna(s) or str(s).strip() == 'None':
        return set()
    tokens = str(s).split()
    return {int(t) for t in tokens if t != 'None'}

ref_df['product_set'] = ref_df['product_id'].apply(parse_products)
ref_pred = ref_df.set_index('order_id')['product_set'].to_dict()

print(f'\nFaron 예측 주문 수: {len(ref_pred):,}')
print(f'Faron 빈 예측(None) 주문 수: {sum(1 for v in ref_pred.values() if len(v) == 0):,}')

In [ ]:
# 전체 주문 ID 기준으로 정렬
all_order_ids = sorted(ref_pred.keys())

# 우리 파일에 없는 주문 → 빈 set
missing_orders = [oid for oid in all_order_ids if oid not in our_pred]
print(f'전체 test 주문 수      : {len(all_order_ids):,}')
print(f'우리가 예측한 주문 수  : {len(our_pred):,}')
print(f'우리가 빈 예측인 주문  : {len(missing_orders):,}')

for oid in missing_orders:
    our_pred[oid] = set()  # 빈 set으로 채움

print('\n샘플 비교 (첫 3개 주문):')
for oid in all_order_ids[:3]:
    print(f'  order {oid}')
    print(f'    우리  : {sorted(our_pred[oid])[:8]}  ({len(our_pred[oid])}개)')
    print(f'    Faron : {sorted(ref_pred[oid])[:8]}  ({len(ref_pred[oid])}개)')

---
## 2. 주문별 F1-Score 계산 (Instacart 공식 지표)

In [ ]:
def order_metrics(pred_set, ref_set):
    """주문 1개의 Precision / Recall / F1 / Jaccard 계산"""
    if len(pred_set) == 0 and len(ref_set) == 0:
        return 1.0, 1.0, 1.0, 1.0   # 둘 다 빈 예측 → 완전 일치
    if len(pred_set) == 0 or len(ref_set) == 0:
        return 0.0, 0.0, 0.0, 0.0   # 한쪽만 빔

    intersection = len(pred_set & ref_set)
    precision  = intersection / len(pred_set)
    recall     = intersection / len(ref_set)
    f1         = (2 * precision * recall / (precision + recall)
                  if (precision + recall) > 0 else 0.0)
    jaccard    = intersection / len(pred_set | ref_set)
    return precision, recall, f1, jaccard


print('주문별 지표 계산 중...')
rows = []
for oid in all_order_ids:
    p, r, f1, jac = order_metrics(our_pred[oid], ref_pred[oid])
    rows.append({
        'order_id'      : oid,
        'our_count'     : len(our_pred[oid]),
        'ref_count'     : len(ref_pred[oid]),
        'intersection'  : len(our_pred[oid] & ref_pred[oid]),
        'precision'     : p,
        'recall'        : r,
        'f1'            : f1,
        'jaccard'       : jac,
    })

metrics_df = pd.DataFrame(rows)
print('계산 완료')
metrics_df.head()

---
## 3. 전체 집계 결과

In [ ]:
SEP = '=' * 52
print(SEP)
print('  우리 모델 vs Faron-opt_bagging-v3  비교 결과')
print(SEP)
print(f'비교 주문 수          : {len(metrics_df):,}')
print()
print(f'Mean F1-Score         : {metrics_df["f1"].mean():.4f}')
print(f'Mean Precision        : {metrics_df["precision"].mean():.4f}')
print(f'Mean Recall           : {metrics_df["recall"].mean():.4f}')
print(f'Mean Jaccard          : {metrics_df["jaccard"].mean():.4f}')
print()
print(f'F1 중앙값 (Median)    : {metrics_df["f1"].median():.4f}')
print(f'F1 >= 0.5 주문 비율   : {(metrics_df["f1"] >= 0.5).mean()*100:.1f}%')
print(f'F1 = 1.0 완전 일치    : {(metrics_df["f1"] == 1.0).sum():,}건  '
      f'({(metrics_df["f1"]==1.0).mean()*100:.1f}%)')
print(f'F1 = 0.0 완전 불일치  : {(metrics_df["f1"] == 0.0).sum():,}건  '
      f'({(metrics_df["f1"]==0.0).mean()*100:.1f}%)')
print()
print(f'우리 평균 예측 수     : {metrics_df["our_count"].mean():.2f}개/주문')
print(f'Faron 평균 예측 수    : {metrics_df["ref_count"].mean():.2f}개/주문')
print(f'평균 교집합 수        : {metrics_df["intersection"].mean():.2f}개/주문')
print(SEP)

---
## 4. 시각화

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# 1) F1-Score 분포
axes[0,0].hist(metrics_df['f1'], bins=50, color='tomato', alpha=0.8, edgecolor='white')
axes[0,0].axvline(metrics_df['f1'].mean(), color='black', linewidth=2,
                  label=f'평균 {metrics_df["f1"].mean():.4f}')
axes[0,0].axvline(metrics_df['f1'].median(), color='navy', linewidth=1.5,
                  linestyle='--', label=f'중앙값 {metrics_df["f1"].median():.4f}')
axes[0,0].set_title('주문별 F1-Score 분포', fontsize=12)
axes[0,0].set_xlabel('F1-Score')
axes[0,0].set_ylabel('주문 수')
axes[0,0].legend()

# 2) Precision vs Recall 산점도
sample = metrics_df.sample(min(3000, len(metrics_df)), random_state=42)
axes[0,1].scatter(sample['precision'], sample['recall'],
                  alpha=0.2, s=8, color='steelblue')
axes[0,1].axline((0,0), slope=1, color='black', linewidth=0.8, linestyle='--', label='P=R')
axes[0,1].set_title('Precision vs Recall (샘플 3,000건)', fontsize=12)
axes[0,1].set_xlabel('Precision')
axes[0,1].set_ylabel('Recall')
axes[0,1].set_xlim(0, 1.05)
axes[0,1].set_ylim(0, 1.05)
axes[0,1].legend()

# 3) 예측 상품 수 비교
axes[0,2].scatter(sample['ref_count'], sample['our_count'],
                  alpha=0.2, s=8, color='mediumseagreen')
max_cnt = max(metrics_df['ref_count'].max(), metrics_df['our_count'].max())
axes[0,2].axline((0,0), slope=1, color='black', linewidth=0.8, linestyle='--', label='동일 수')
axes[0,2].set_title('예측 상품 수 비교', fontsize=12)
axes[0,2].set_xlabel('Faron 예측 수')
axes[0,2].set_ylabel('우리 예측 수')
axes[0,2].legend()

# 4) Faron 예측 수별 우리 F1 평균
metrics_df['ref_count_bin'] = pd.cut(metrics_df['ref_count'],
                                     bins=[0,2,4,6,8,10,15,20,100],
                                     labels=['1-2','3-4','5-6','7-8','9-10','11-15','16-20','21+'])
f1_by_ref = metrics_df.groupby('ref_count_bin', observed=True)['f1'].mean()
axes[1,0].bar(f1_by_ref.index.astype(str), f1_by_ref.values, color='mediumpurple')
for i, v in enumerate(f1_by_ref.values):
    axes[1,0].text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=8)
axes[1,0].set_title('Faron 예측 수 구간별 평균 F1', fontsize=12)
axes[1,0].set_xlabel('Faron 예측 상품 수')
axes[1,0].set_ylabel('평균 F1-Score')

# 5) F1 구간별 주문 수 (막대)
bins = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.01]
labels = ['0~0.1','0.1~0.2','0.2~0.3','0.3~0.4','0.4~0.5',
          '0.5~0.6','0.6~0.7','0.7~0.8','0.8~0.9','0.9~1.0','1.0']
metrics_df['f1_bin'] = pd.cut(metrics_df['f1'], bins=bins, labels=labels, right=False)
f1_dist = metrics_df['f1_bin'].value_counts().sort_index()
colors_bar = ['#d73027' if l in ['0~0.1','0.1~0.2','0.2~0.3'] else
              '#fee090' if l in ['0.3~0.4','0.4~0.5','0.5~0.6'] else
              '#91cf60' for l in f1_dist.index.astype(str)]
axes[1,1].bar(f1_dist.index.astype(str), f1_dist.values, color=colors_bar, edgecolor='white')
axes[1,1].set_title('F1-Score 구간별 주문 수', fontsize=12)
axes[1,1].set_xlabel('F1-Score 구간')
axes[1,1].set_ylabel('주문 수')
axes[1,1].tick_params(axis='x', rotation=45)

# 6) Jaccard 분포
axes[1,2].hist(metrics_df['jaccard'], bins=50, color='steelblue', alpha=0.8, edgecolor='white')
axes[1,2].axvline(metrics_df['jaccard'].mean(), color='black', linewidth=2,
                  label=f'평균 {metrics_df["jaccard"].mean():.4f}')
axes[1,2].set_title('주문별 Jaccard 유사도 분포', fontsize=12)
axes[1,2].set_xlabel('Jaccard')
axes[1,2].set_ylabel('주문 수')
axes[1,2].legend()

plt.suptitle('우리 모델 vs Faron-opt_bagging-v3 비교', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 5. 상세 분석 — 잘 맞은 주문 / 못 맞은 주문

In [ ]:
# 상위 10개 (가장 일치) vs 하위 10개 (가장 불일치)
top10    = metrics_df.nlargest(10,  'f1')[['order_id','our_count','ref_count','intersection','precision','recall','f1']]
bottom10 = metrics_df[metrics_df['ref_count'] > 0].nsmallest(10, 'f1')[
    ['order_id','our_count','ref_count','intersection','precision','recall','f1']
]

print('=== F1 상위 10개 주문 (우리 ≈ Faron) ===')
print(top10.to_string(index=False))

print('\n=== F1 하위 10개 주문 (우리 ≠ Faron) ===')
print(bottom10.to_string(index=False))

In [ ]:
# 특정 주문 상세 비교 — F1이 낮은 주문 1개 선택
bad_order_id = bottom10['order_id'].iloc[0]
our_set = our_pred[bad_order_id]
ref_set = ref_pred[bad_order_id]

only_us   = our_set - ref_set   # 우리만 예측
only_faron = ref_set - our_set  # Faron만 예측
both      = our_set & ref_set   # 공통 예측

SEP = '-' * 45
print(f'=== order_id={bad_order_id} 상세 비교 ===')
print(f'공통 예측    ({len(both):>2}개): {sorted(both)}')
print(f'우리만 예측  ({len(only_us):>2}개): {sorted(only_us)}')
print(f'Faron만 예측 ({len(only_faron):>2}개): {sorted(only_faron)}')

---
## 6. 공통/고유 예측 상품 통계

In [ ]:
# 전체 주문에 걸쳐 상품별 예측 빈도 비교
from collections import Counter

our_all   = Counter(pid for pset in our_pred.values() for pid in pset)
ref_all   = Counter(pid for pset in ref_pred.values() for pid in pset)

our_total = sum(our_all.values())
ref_total = sum(ref_all.values())

print(f'우리 총 예측 상품 수 (중복 포함): {our_total:,}')
print(f'Faron 총 예측 상품 수 (중복 포함): {ref_total:,}')
print(f'차이: {our_total - ref_total:+,}  ({(our_total/ref_total - 1)*100:+.1f}%)')

# 공통으로 예측된 고유 상품 수
common_products = set(our_all.keys()) & set(ref_all.keys())
only_our_products   = set(our_all.keys()) - set(ref_all.keys())
only_faron_products = set(ref_all.keys()) - set(our_all.keys())

print(f'\n고유 상품 기준:')
print(f'  우리만 예측한 상품 수   : {len(only_our_products):,}')
print(f'  Faron만 예측한 상품 수  : {len(only_faron_products):,}')
print(f'  공통으로 예측한 상품 수 : {len(common_products):,}')

In [ ]:
# 두 모델이 모두 자주 예측한 상위 상품 (공통 인기 상품)
product_compare = pd.DataFrame({
    'product_id'    : list(common_products),
    'our_freq'      : [our_all[p] for p in common_products],
    'faron_freq'    : [ref_all[p]  for p in common_products],
})
product_compare['total_freq'] = product_compare['our_freq'] + product_compare['faron_freq']
product_compare = product_compare.sort_values('total_freq', ascending=False)

print('=== 두 모델이 공통으로 가장 많이 예측한 상위 20개 상품 ===')
print(product_compare.head(20).to_string(index=False))

---
## 7. 최종 요약

In [ ]:
mean_f1  = metrics_df['f1'].mean()
mean_p   = metrics_df['precision'].mean()
mean_r   = metrics_df['recall'].mean()
mean_jac = metrics_df['jaccard'].mean()
perfect  = (metrics_df['f1'] == 1.0).mean()
zero_f1  = (metrics_df['f1'] == 0.0).mean()
above_half = (metrics_df['f1'] >= 0.5).mean()

SEP = '=' * 52
print(SEP)
print('  최종 비교 요약')
print(SEP)
print(f'  Mean F1-Score     : {mean_f1:.4f}')
print(f'  Mean Precision    : {mean_p:.4f}')
print(f'  Mean Recall       : {mean_r:.4f}')
print(f'  Mean Jaccard      : {mean_jac:.4f}')
print()
print(f'  완전 일치 (F1=1.0): {perfect*100:.1f}%')
print(f'  F1 >= 0.5        : {above_half*100:.1f}%')
print(f'  완전 불일치(F1=0) : {zero_f1*100:.1f}%')
print()

if mean_f1 >= 0.6:
    grade = '우수 — Faron 모델과 높은 일치도'
elif mean_f1 >= 0.4:
    grade = '양호 — 절반 이상의 상품이 일치'
elif mean_f1 >= 0.25:
    grade = '보통 — 일부 개선 필요'
else:
    grade = '개선 필요 — 예측 경향이 크게 다름'

print(f'  평가: {grade}')
print(SEP)
print()
print('※ 이 F1-Score는 Faron 참조 모델과의 일치도이며,')
print('   실제 정답과의 정확도(Kaggle LB)와는 다를 수 있습니다.')